In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import pymannkendall as mk
import data_loader as dl


In [2]:
def get_project_root() -> Path:
    cwd = Path.cwd().resolve()
    return cwd.parent if cwd.name == "scripts" else cwd


base_dir = get_project_root()
github_ready_dir = base_dir
data_raw_dir = base_dir / "data_raw"
out_dir = github_ready_dir / "data_intermediate"
out_dir.mkdir(parents=True, exist_ok=True)

required_raw = data_raw_dir / "Thiessen_Polygons_100yrs.csv"
if not required_raw.exists():
    raise FileNotFoundError(f"Required data file not found: {required_raw}")


In [3]:
df_neon = pd.read_csv(data_raw_dir / "Thiessen_Polygons_100yrs.csv")
df_neon = df_neon[["ID_1", "Lat", "Lon", "Area_sqkm"]]
df_neon.columns = ["ID", "Lat", "Lon", "Area"]


In [4]:
# ============================================================
# CONUS MK Trend Analysis for PRCP, TMAX, TMIN (and others)
# ============================================================


# ============================================================
# Helper functions
# ============================================================

def compute_water_year(date_series: pd.Series) -> pd.Series:
    """
    Compute water year from a pandas Series of datetime objects.
    Water year starts in October.
    Example: Oct 1924 → WY 1925.
    """
    dates = pd.to_datetime(date_series)
    return dates.apply(lambda d: d.year + 1 if d.month >= 10 else d.year)


def clip_spei_spi(df: pd.DataFrame,
                  lower_bound: float = -3.0,
                  upper_bound: float = 3.0) -> pd.DataFrame:
    """
    Clip all columns containing 'spei' or 'spi' in their name
    to [lower_bound, upper_bound].
    """
    spei_spi_cols = [
        col for col in df.columns
        if "spei" in col.lower() or "spi" in col.lower()
    ]
    for col in spei_spi_cols:
        df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)
    return df


# ============================================================
# Modified Mann–Kendall + Sen's slope
# ============================================================

def modified_mk_test(series, alpha: float = 0.05) -> dict:
    """
    Run Hamed & Rao Modified Mann–Kendall test on a 1D series.
    NaNs are dropped.

    Returns a dict with:
      trend, p, slope, h, z, tau, s, var_s, intercept.
    """
    s = pd.Series(series).astype(float).dropna()
    res = mk.hamed_rao_modification_test(s, alpha=alpha)

    return {
        "trend": res.trend,
        "p": res.p,
        "slope": res.slope,
        "h": res.h,
        "z": res.z,
        "tau": res.Tau,
        "s": res.s,
        "var_s": res.var_s,
        "intercept": res.intercept,
    }


def mk_for_dataframe(
    df: pd.DataFrame,
    value_cols,
    group_col: str = "ID",
    time_col: str | None = None,
    alpha: float = 0.05,
) -> pd.DataFrame:
    """
    Apply Modified Mann–Kendall + Sen's slope to multiple columns,
    grouped by group_col.

    Returns DataFrame with:
      [group_col, variable, trend, p, slope, h, z, tau, s, var_s, intercept, n]
    """
    if time_col is not None:
        df = df.sort_values([group_col, time_col])

    records = []

    for gid, g in df.groupby(group_col):
        for col in value_cols:
            series = g[col].values
            res = modified_mk_test(series, alpha=alpha)
            records.append(
                {
                    group_col: gid,
                    "variable": col,
                    "trend": res["trend"],
                    "p": res["p"],
                    "slope": res["slope"],
                    "h": res["h"],
                    "z": res["z"],
                    "tau": res["tau"],
                    "s": res["s"],
                    "var_s": res["var_s"],
                    "intercept": res["intercept"],
                    "n": len(series),
                }
            )

    return pd.DataFrame.from_records(records)


def mk_two_timeframes(
    df: pd.DataFrame,
    value_cols,
    group_col: str = "ID",
    time_col: str = "W_YEAR",
    alpha: float = 0.05,
    full_start: int = 1926,
    full_end: int = 2025,
    recent_start: int = 1996,
    recent_end: int = 2025,
) -> pd.DataFrame:
    """
    Run Modified MK + Sen's slope for two windows:
      - full (e.g., 1926–2025)
      - recent (e.g., 1996–2025)

    Returns stacked DataFrame with extra columns:
      window, window_type.
    """
    # Full / ~100-yr window
    df_full = df[(df[time_col] >= full_start) & (df[time_col] <= full_end)].copy()
    res_full = mk_for_dataframe(
        df_full,
        value_cols=value_cols,
        group_col=group_col,
        time_col=time_col,
        alpha=alpha,
    )
    res_full["window"] = f"{full_start}-{full_end}"
    res_full["window_type"] = f"full_{full_end - full_start + 1}yr"

    # Recent / ~30-yr window
    df_recent = df[(df[time_col] >= recent_start) & (df[time_col] <= recent_end)].copy()
    res_recent = mk_for_dataframe(
        df_recent,
        value_cols=value_cols,
        group_col=group_col,
        time_col=time_col,
        alpha=alpha,
    )
    res_recent["window"] = f"{recent_start}-{recent_end}"
    res_recent["window_type"] = f"recent_{recent_end - recent_start + 1}yr"

    return pd.concat([res_full, res_recent], ignore_index=True)


# ============================================================
# Ecoregion + CONUS summaries
# ============================================================

def summarize_trends_by_ecoregion(
    mk_df: pd.DataFrame,
    area_df: pd.DataFrame,
    variable: str = "PRCP",
    id_col: str = "ID",
    domain_col: str = "domainName",
    area_col: str = "Area",
    slope_factor: float = 10.0,  # 10 → convert slope/yr to slope/decade
) -> pd.DataFrame:
    """
    Summarize MK trend results by ecoregion (domainName) and for CONUS
    (all domains combined), for each window_type and trend_dir.

    Returns DataFrame with:
      [domain_col, window, window_type, trend_dir,
       PA, slope_min, slope_max, slope_mean, slope_median, n_sig]
    """
    mk_var = mk_df[mk_df["variable"] == variable].copy()

    merged = mk_var.merge(
        area_df[[id_col, domain_col, area_col]],
        on=id_col,
        how="left",
    )

    records = []
    trend_dirs = ["increasing", "decreasing"]

    # ---------- 1) DOMAIN-LEVEL (each ecoregion) ----------
    for (dom, win_type, win), g in merged.groupby([domain_col, "window_type", "window"]):
        total_area = g[area_col].sum()

        for tdir in trend_dirs:
            g_tr = g[(g["h"] == True) & (g["trend"] == tdir)]

            sig_area = g_tr[area_col].sum()
            PA = 100.0 * sig_area / total_area if total_area > 0 else np.nan

            slopes = g_tr["slope"] * slope_factor  # e.g., mm/decade or °C/decade

            if slopes.empty:
                s_min = s_max = s_mean = s_med = np.nan
            else:
                s_min = slopes.min()
                s_max = slopes.max()
                s_mean = slopes.mean()
                s_med = slopes.median()

            records.append(
                {
                    domain_col: dom,
                    "window": win,
                    "window_type": win_type,
                    "trend_dir": tdir,
                    "PA": PA,
                    "slope_min": s_min,
                    "slope_max": s_max,
                    "slope_mean": s_mean,
                    "slope_median": s_med,
                    "n_sig": len(g_tr),
                }
            )

    # ---------- 2) CONUS-LEVEL (all domains combined) ----------
    for (win_type, win), g in merged.groupby(["window_type", "window"]):
        total_area = g[area_col].sum()

        for tdir in trend_dirs:
            g_tr = g[(g["h"] == True) & (g["trend"] == tdir)]

            sig_area = g_tr[area_col].sum()
            PA = 100.0 * sig_area / total_area if total_area > 0 else np.nan

            slopes = g_tr["slope"] * slope_factor

            if slopes.empty:
                s_min = s_max = s_mean = s_med = np.nan
            else:
                s_min = slopes.min()
                s_max = slopes.max()
                s_mean = slopes.mean()
                s_med = slopes.median()

            records.append(
                {
                    domain_col: "CONUS",
                    "window": win,
                    "window_type": win_type,
                    "trend_dir": tdir,
                    "PA": PA,
                    "slope_min": s_min,
                    "slope_max": s_max,
                    "slope_mean": s_mean,
                    "slope_median": s_med,
                    "n_sig": len(g_tr),
                }
            )

    return pd.DataFrame.from_records(records)


def make_grouped_pivot(summary_df: pd.DataFrame,
                       domain_col: str = "domainName") -> pd.DataFrame:
    """
    Create a pivot table where all 30-yr columns are grouped together
    and all 100-yr columns are grouped together.
    Includes the CONUS row naturally as another 'domain'.
    """
    pivot = summary_df.pivot_table(
        index=domain_col,
        columns=["window_type", "trend_dir"],
        values=["PA", "slope_min", "slope_max", "slope_mean", "slope_median"],
    )

    ordered_cols = [
        ("PA", "recent_30yr", "increasing"),
        ("PA", "recent_30yr", "decreasing"),
        ("slope_min", "recent_30yr", "increasing"),
        ("slope_max", "recent_30yr", "increasing"),
        ("slope_mean", "recent_30yr", "increasing"),
        ("slope_median", "recent_30yr", "increasing"),
        ("slope_min", "recent_30yr", "decreasing"),
        ("slope_max", "recent_30yr", "decreasing"),
        ("slope_mean", "recent_30yr", "decreasing"),
        ("slope_median", "recent_30yr", "decreasing"),
        ("PA", "full_100yr", "increasing"),
        ("PA", "full_100yr", "decreasing"),
        ("slope_min", "full_100yr", "increasing"),
        ("slope_max", "full_100yr", "increasing"),
        ("slope_mean", "full_100yr", "increasing"),
        ("slope_median", "full_100yr", "increasing"),
        ("slope_min", "full_100yr", "decreasing"),
        ("slope_max", "full_100yr", "decreasing"),
        ("slope_mean", "full_100yr", "decreasing"),
        ("slope_median", "full_100yr", "decreasing"),
    ]

    # Keep only columns that actually exist
    ordered_cols = [c for c in ordered_cols if c in pivot.columns]
    pivot = pivot.reindex(columns=ordered_cols)

    return pivot


# ============================================================
# Run summaries for each variable and save
# ============================================================

def summarize_and_save_for_variable(
    mk_two: pd.DataFrame,
    df_neon_area: pd.DataFrame,
    variable: str,
    out_dir: Path,
    id_col: str = "ID",
    domain_col: str = "domainName",
    area_col: str = "Area",
    slope_factor: float = 10.0,
) -> None:
    """
    Build ecoregion + CONUS summary and pivot for one variable,
    then save to CSV.
    """
    print(f"\n=== Summarizing trends for variable: {variable} ===")

    summary_var = summarize_trends_by_ecoregion(
        mk_df=mk_two,
        area_df=df_neon_area,
        variable=variable,
        id_col=id_col,
        domain_col=domain_col,
        area_col=area_col,
        slope_factor=slope_factor,
    )

    # Optional rounding to make tables cleaner
    cols_to_round = [c for c in summary_var.columns
                     if ("PA" in c) or ("slope_" in c)]
    summary_var[cols_to_round] = summary_var[cols_to_round].round(4)

    pivot_var = make_grouped_pivot(summary_var, domain_col=domain_col)

    var_lower = variable.lower()
    summary_path = out_dir / f"summary_{var_lower}_by_domain_conus.csv"
    pivot_path = out_dir / f"pivot_{var_lower}_for_word_conus.csv"

    summary_var.to_csv(summary_path, index=False)
    pivot_var.to_csv(pivot_path)

    print(f"  Saved summary -> {summary_path}")
    print(f"  Saved pivot   -> {pivot_path}")


# ============================================================
# Main analysis pipeline
# ============================================================

def main() -> None:
    # -----------------------------------------
    # Paths
    # -----------------------------------------
    base_dir = find_project_root()
    github_ready_dir = find_github_ready_dir()
    data_raw_dir = base_dir / "data_raw"
    out_dir = github_ready_dir / "data_intermediate"
    out_dir.mkdir(parents=True, exist_ok=True)

    # -----------------------------------------
    # Load daily & monthly data
    df_daily = dl.df_daily.copy()
    df_monthly = dl.df_monthly_updated.copy()
    # -----------------------------------------
    # Load Thiessen polygons / area info
    # -----------------------------------------
    df_neon = pd.read_csv(data_raw_dir / "Thiessen_Polygons_100yrs.csv")
    df_neon = df_neon[["ID_1", "Lat", "Lon", "Area_sqkm"]]
    df_neon.columns = ["ID", "Lat", "Lon", "Area"]

    # Ecoregion assignment
    df_id_neon = pd.read_csv(data_raw_dir / "ID_ecoregion.csv")
    df_id_neon_updated = df_id_neon.copy()

    replace_map = {
        "Atlantic Neotropical": "Southeast",
        "Appalachians / Cumberland Plateau": "Appalachians",
        "Southern Rockies / Colorado Plateau": "Colorado Plateau",
    }
    df_id_neon_updated["domainName"] = df_id_neon_updated["domainName"].replace(
        replace_map
    )
    df_id_neon_updated = df_id_neon_updated[["ID", "domainName"]]

    df_neon_area = df_neon[["ID", "Area"]].merge(
        df_id_neon_updated, on="ID", how="inner"
    )

    # -----------------------------------------
    # Annual aggregation by water year

    df_annual = dl.df_yearly.copy()
    # -----------------------------------------
    # Modified MK for two timeframes (runs once for all variables)
    # -----------------------------------------
    value_cols = ["PRCP", "TMAX", "TMIN","AI","PCI","WSDI","CSDI","SDII"]
    mk_two = mk_two_timeframes(
        df_annual,
        value_cols=value_cols,
        group_col="ID",
        time_col="W_YEAR",
        alpha=0.05,
    )

    # Save full MK results (all variables)
    mk_two_out = out_dir / "mk_two_PRCP_TMAX_TMIN_AI.csv"
    mk_two.to_csv(mk_two_out, index=False)
    print(f"\nSaved MK results for all variables -> {mk_two_out}")

    # -----------------------------------------
    # Summaries by ecoregion + CONUS for each variable
    # -----------------------------------------
    for var in value_cols:
        summarize_and_save_for_variable(
            mk_two=mk_two,
            df_neon_area=df_neon_area,
            variable=var,
            out_dir=out_dir,
            id_col="ID",
            domain_col="domainName",
            area_col="Area",
            slope_factor=10.0,  # per decade
        )

    print("\nAll variables processed.")


if __name__ == "__main__":
    main()


NameError: name 'find_project_root' is not defined